# Smart Macros: Clinical Risk Prediction Model (Training & Evaluation)
**Author:** Pratyaksh Gupta | **Module:** 6COSC023W Final Year Project

### Objective (LO1, LO4, LO7)
This notebook demonstrates the generation of a synthetic patient cohort and the training of a **Random Forest Regressor** to predict physiological glycemic and cardiovascular risk. 

To resolve the "Black Box" nature of standard deep learning models, an ensemble method was chosen to support **Explainable AI (XAI)**, aligning with the project's ethical and clinical requirements.

### The Ground Truth Heuristic
The underlying physiological risk is simulated using the following metabolic constraint formula:
$Risk = \beta_1(Carbs) - \beta_2(Fiber) + \beta_3(Sodium) + \alpha(Condition)$

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
import joblib

print("🚀 INITIALIZING SYNTHETIC DATA GENERATION (n=1000)...")

# 1. GENERATE SYNTHETIC CLINICAL COHORT
np.random.seed(42) # For academic reproducibility
n_samples = 1000

# Features: Age, Condition (0=None, 1=Coeliac, 2=Diabetes, 3=CVD)
ages = np.random.randint(18, 80, n_samples)
conditions = np.random.choice([0, 1, 2, 3], n_samples, p=[0.4, 0.1, 0.3, 0.2])
basket_carbs = np.random.uniform(20, 150, n_samples)
basket_fiber = np.random.uniform(0, 30, n_samples)
basket_sodium = np.random.uniform(10, 1500, n_samples) # Added for CVD

# Target: Physiological Risk (1.0 to 10.0)
base_risk = (basket_carbs * 0.04) - (basket_fiber * 0.15) + (basket_sodium * 0.002)

# Clinical Penalties (The hidden logic the AI must learn)
diabetic_penalty = np.where(conditions == 2, (basket_carbs * 0.05), 0)
cvd_penalty = np.where(conditions == 3, (basket_sodium * 0.003), 0)
noise = np.random.normal(0, 0.5, n_samples)

glycemic_risk = np.clip(base_risk + diabetic_penalty + cvd_penalty + noise + 1.0, 1.0, 10.0)

df = pd.DataFrame({
    'age': ages,
    'condition_code': conditions,
    'total_carbs_g': basket_carbs,
    'total_fiber_g': basket_fiber,
    'total_sodium_mg': basket_sodium,
    'clinical_risk_score': glycemic_risk
})

print("✅ Data successfully generated. Here is a sample:")
display(df.head())